# GWAS ADD结果快速筛选与清理准备

目的：

1. 扫描全部 `*.glm.linear.add` / `*.glm.linear.add.gz`
2. 快速提取每个 phenotype 的：
   - `min_p`
   - lead SNP
   - `p < 1e-5 / 1e-6 / 5e-8` 的位点数
3. 生成“值得优先画 Manhattan 图”的候选列表
4. 先不删除原文件，只做 summary，确认后再清理

说明：

- 你现在空间紧，所以第一步不要盲目保留全部 ADD。
- 更稳的办法是先把所有 ADD 扫一遍，得到一个很小的 summary。
- 后续你可以只保留：
  - 有信号的 phenotype 的 `.add`
  - 全局 summary
- 这个 notebook 默认**不做删除**，最后单独提供一个“仅保留候选 phenotype ADD”的可选代码块。

In [2]:
from pathlib import Path
import os
import gzip
import shutil
from concurrent.futures import ProcessPoolExecutor, as_completed

import pandas as pd
from IPython.display import display

# =========================
# 项目路径
# =========================
PROJECT_ROOT = Path("/mnt/zzbnew/peixunban/zhanghebin/CIMA_multiomics_regulation")

OUT_BASE = PROJECT_ROOT / "data/results/gwas_all"
PER_PHENO_DIR = OUT_BASE / "per_pheno"
SUMMARY_DIR = OUT_BASE / "summary"
LOG_DIR = OUT_BASE / "logs"

for d in [SUMMARY_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# =========================
# 扫描参数
# =========================
# 这里只是读文件做 summary，不调用 plink，所以线程可比 GWAS 更高一点
N_WORKERS = 16

# 建议阈值
PLOT_STRONG_P = 1e-6      # 很值得优先画图
PLOT_SUGGESTIVE_P = 1e-5  # 建议性显著，值得看
GENOME_WIDE_P = 5e-8      # 传统 genome-wide

# 输出文件
SUMMARY_TSV = SUMMARY_DIR / "add_signal_summary.tsv"
PLOT_CANDIDATE_TSV = SUMMARY_DIR / "add_plot_candidates.tsv"
TOP_PHENO_TSV = SUMMARY_DIR / "add_top_phenotypes_by_minp.tsv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PER_PHENO_DIR:", PER_PHENO_DIR)
print("SUMMARY_TSV:", SUMMARY_TSV)
print("PLOT_CANDIDATE_TSV:", PLOT_CANDIDATE_TSV)
print("TOP_PHENO_TSV:", TOP_PHENO_TSV)
print("N_WORKERS:", N_WORKERS)

PROJECT_ROOT: /mnt/zzbnew/peixunban/zhanghebin/CIMA_multiomics_regulation
PER_PHENO_DIR: /mnt/zzbnew/peixunban/zhanghebin/CIMA_multiomics_regulation/data/results/gwas_all/per_pheno
SUMMARY_TSV: /mnt/zzbnew/peixunban/zhanghebin/CIMA_multiomics_regulation/data/results/gwas_all/summary/add_signal_summary.tsv
PLOT_CANDIDATE_TSV: /mnt/zzbnew/peixunban/zhanghebin/CIMA_multiomics_regulation/data/results/gwas_all/summary/add_plot_candidates.tsv
TOP_PHENO_TSV: /mnt/zzbnew/peixunban/zhanghebin/CIMA_multiomics_regulation/data/results/gwas_all/summary/add_top_phenotypes_by_minp.tsv
N_WORKERS: 16


## Step 1. 扫描 ADD 文件

兼容两种情况：

- `*.glm.linear.add`
- `*.glm.linear.add.gz`

phenotype 名默认取父目录名。

In [3]:
def list_add_files(per_pheno_dir: Path):
    files = []
    for f in per_pheno_dir.glob("*/*.glm.linear.add"):
        if f.is_file():
            files.append(f)
    for f in per_pheno_dir.glob("*/*.glm.linear.add.gz"):
        if f.is_file():
            files.append(f)
    files = sorted(set(files))
    return files

add_files = list_add_files(PER_PHENO_DIR)
print("ADD file count:", len(add_files))
display(pd.DataFrame({"add_file": [str(x) for x in add_files[:10]]}))

ADD file count: 1532


,add_file
0,/mnt/zzbnew/peixunban/zhanghebin/CIMA_multiomi...
1,/mnt/zzbnew/peixunban/zhanghebin/CIMA_multiomi...
2,/mnt/zzbnew/peixunban/zhanghebin/CIMA_multiomi...
3,/mnt/zzbnew/peixunban/zhanghebin/CIMA_multiomi...
4,/mnt/zzbnew/peixunban/zhanghebin/CIMA_multiomi...
5,/mnt/zzbnew/peixunban/zhanghebin/CIMA_multiomi...
6,/mnt/zzbnew/peixunban/zhanghebin/CIMA_multiomi...
7,/mnt/zzbnew/peixunban/zhanghebin/CIMA_multiomi...
8,/mnt/zzbnew/peixunban/zhanghebin/CIMA_multiomi...
9,/mnt/zzbnew/peixunban/zhanghebin/CIMA_multiomi...


## Step 2. 定义单文件解析函数

只读取必要列，避免占内存。

输出字段包括：

- phenotype
- file
- is_gz
- n_rows
- n_non_na_p
- min_p
- lead SNP 信息
- 小于不同阈值的命中数

In [4]:
USECOLS = ["#CHROM", "POS", "ID", "P", "BETA", "SE", "ERRCODE"]

def read_add_file(add_file: Path) -> pd.DataFrame:
    return pd.read_csv(
        add_file,
        sep="\t",
        compression="infer",
        usecols=lambda c: c in USECOLS,
        low_memory=False
    )

def summarize_add_file(add_file: Path) -> dict:
    phenotype = add_file.parent.name
    out = {
        "phenotype": phenotype,
        "file": str(add_file),
        "is_gz": str(add_file).endswith(".gz"),
        "exists": add_file.exists(),
        "file_size_mb": round(add_file.stat().st_size / 1024 / 1024, 4) if add_file.exists() else None,
        "status": "ok",
        "error": None,
        "n_rows": None,
        "n_non_na_p": None,
        "min_p": None,
        "lead_chr": None,
        "lead_pos": None,
        "lead_id": None,
        "lead_beta": None,
        "lead_se": None,
        "lead_errcode": None,
        "n_p_lt_1e5": 0,
        "n_p_lt_1e6": 0,
        "n_p_lt_5e8": 0,
        "p01": None,
        "p001": None,
        "p0001": None,
    }
    try:
        df = read_add_file(add_file)
        out["n_rows"] = len(df)

        if "P" not in df.columns:
            out["status"] = "failed"
            out["error"] = "missing P column"
            return out

        df["P"] = pd.to_numeric(df["P"], errors="coerce")
        out["n_non_na_p"] = int(df["P"].notna().sum())

        if out["n_non_na_p"] == 0:
            out["status"] = "failed"
            out["error"] = "no valid P"
            return out

        valid = df[df["P"].notna()].copy()
        valid = valid[valid["P"] > 0].copy()

        if len(valid) == 0:
            out["status"] = "failed"
            out["error"] = "all P <= 0 or invalid"
            return out

        out["min_p"] = float(valid["P"].min())
        out["n_p_lt_1e5"] = int((valid["P"] < 1e-5).sum())
        out["n_p_lt_1e6"] = int((valid["P"] < 1e-6).sum())
        out["n_p_lt_5e8"] = int((valid["P"] < 5e-8).sum())
        out["p01"] = float(valid["P"].quantile(0.01))
        out["p001"] = float(valid["P"].quantile(0.001))
        out["p0001"] = float(valid["P"].quantile(0.0001))

        lead = valid.sort_values("P", ascending=True).iloc[0]
        out["lead_chr"] = lead["#CHROM"] if "#CHROM" in lead.index else None
        out["lead_pos"] = lead["POS"] if "POS" in lead.index else None
        out["lead_id"] = lead["ID"] if "ID" in lead.index else None
        out["lead_beta"] = lead["BETA"] if "BETA" in lead.index else None
        out["lead_se"] = lead["SE"] if "SE" in lead.index else None
        out["lead_errcode"] = lead["ERRCODE"] if "ERRCODE" in lead.index else None
        return out

    except Exception as e:
        out["status"] = "failed"
        out["error"] = repr(e)
        return out

## Step 3. 并行汇总全部 phenotype

这一步只读 ADD，不调用 PLINK，因此比 GWAS 本身轻很多。

In [5]:
results = []

with ProcessPoolExecutor(max_workers=N_WORKERS) as ex:
    fut_map = {ex.submit(summarize_add_file, f): f for f in add_files}
    for i, fut in enumerate(as_completed(fut_map), 1):
        res = fut.result()
        results.append(res)
        if i % 50 == 0 or i == len(fut_map):
            print(f"[{i}/{len(fut_map)}] done")

summary_df = pd.DataFrame(results)
summary_df = summary_df.sort_values(["status", "min_p"], ascending=[True, True], na_position="last").reset_index(drop=True)

summary_df.to_csv(SUMMARY_TSV, sep="\t", index=False)
print("saved:", SUMMARY_TSV)
print("shape:", summary_df.shape)

display(summary_df.head(20))

[50/1532] done
[100/1532] done
[150/1532] done
[200/1532] done
[250/1532] done
[300/1532] done
[350/1532] done
[400/1532] done
[450/1532] done
[500/1532] done
[550/1532] done
[600/1532] done
[650/1532] done
[700/1532] done
[750/1532] done
[800/1532] done
[850/1532] done
[900/1532] done
[950/1532] done
[1000/1532] done
[1050/1532] done
[1100/1532] done
[1150/1532] done
[1200/1532] done
[1250/1532] done
[1300/1532] done
[1350/1532] done
[1400/1532] done
[1450/1532] done
[1500/1532] done
[1532/1532] done
saved: /mnt/zzbnew/peixunban/zhanghebin/CIMA_multiomics_regulation/data/results/gwas_all/summary/add_signal_summary.tsv
shape: (1532, 22)


,phenotype,file,is_gz,exists,file_size_mb,status,error,n_rows,n_non_na_p,min_p,...,lead_id,lead_beta,lead_se,lead_errcode,n_p_lt_1e5,n_p_lt_1e6,n_p_lt_5e8,p01,p001,p0001
0,PE_18_2_22_2,/mnt/zzbnew/peixunban/zhanghebin/CIMA_multiomi...,False,True,685.2066,ok,None,7405576,7299768,5.459910e-10,...,chr8_114969281,0.684860,0.103268,.,336,197,82,0.009954,0.000998,0.000072
1,PC_20_2_22_4,/mnt/zzbnew/peixunban/zhanghebin/CIMA_multiomi...,False,True,686.0261,ok,None,7405576,7322675,9.325720e-10,...,chr3_3078681,0.591835,0.091940,.,103,15,5,0.010281,0.001001,0.000091
2,TAG52_2_FA20_0,/mnt/zzbnew/peixunban/zhanghebin/CIMA_multiomi...,False,True,692.1581,ok,None,7405576,7369195,1.236100e-09,...,chr21_31858140,-0.501662,0.080279,.,83,15,6,0.009386,0.000955,0.000093
3,PE_18_1_18_4,/mnt/zzbnew/peixunban/zhanghebin/CIMA_multiomi...,False,True,684.7895,ok,None,7405576,7288503,1.690420e-09,...,chr5_142589362,-0.673385,0.104125,.,172,70,4,0.010172,0.000885,0.000076
4,P_3_Phosphoglyceric_acid,/mnt/zzbnew/peixunban/zhanghebin/CIMA_multiomi...,False,True,692.3035,ok,None,7405576,7372921,1.996020e-09,...,chr2_153646482,-0.654992,0.106382,.,198,84,8,0.009733,0.000871,0.000067
5,TAG58_7_FA22_5,/mnt/zzbnew/peixunban/zhanghebin/CIMA_multiomi...,False,True,692.2580,ok,None,7405576,7372062,2.190780e-09,...,chr8_72746337,0.520196,0.084689,.,106,49,1,0.009872,0.000947,0.000074
6,PE_16_1_20_4,/mnt/zzbnew/peixunban/zhanghebin/CIMA_multiomi...,False,True,685.1381,ok,None,7405576,7296193,2.367650e-09,...,chr10_96153621,0.932634,0.146568,.,129,36,10,0.009555,0.000991,0.000119
7,PE_16_0_16_0,/mnt/zzbnew/peixunban/zhanghebin/CIMA_multiomi...,False,True,689.4960,ok,None,7405576,7343661,2.738120e-09,...,chr20_63119632,0.574238,0.093046,.,98,13,11,0.010231,0.001066,0.000088
8,TAG54_4_FA18_2,/mnt/zzbnew/peixunban/zhanghebin/CIMA_multiomi...,False,True,692.3917,ok,None,7405576,7372921,2.773820e-09,...,chr11_65246264,1.982260,0.324939,.,22,2,1,0.010153,0.001052,0.000103
9,L_Methionine,/mnt/zzbnew/peixunban/zhanghebin/CIMA_multiomi...,False,True,692.4283,ok,None,7405576,7372921,3.087140e-09,...,chr14_50978200,-0.437916,0.072024,.,153,35,10,0.009654,0.000855,0.000073


## Step 4. 看整体分布

这里先回答几个最实际的问题：

- 一共有多少 phenotype 扫到了？
- 有多少 phenotype 至少有一个 `p < 1e-5`
- 有多少 phenotype 至少有一个 `p < 1e-6`
- 有多少 phenotype 达到 `5e-8`

In [6]:
ok_df = summary_df[summary_df["status"] == "ok"].copy()

print("total ok phenotypes:", len(ok_df))
print("phenotypes with min_p < 1e-5 :", int((ok_df["min_p"] < 1e-5).sum()))
print("phenotypes with min_p < 1e-6 :", int((ok_df["min_p"] < 1e-6).sum()))
print("phenotypes with min_p < 5e-8 :", int((ok_df["min_p"] < 5e-8).sum()))
print("phenotypes with >=1 hit p<1e-5:", int((ok_df["n_p_lt_1e5"] > 0).sum()))
print("phenotypes with >=1 hit p<1e-6:", int((ok_df["n_p_lt_1e6"] > 0).sum()))
print("phenotypes with >=1 hit p<5e-8:", int((ok_df["n_p_lt_5e8"] > 0).sum()))

display(
    ok_df[[
        "phenotype", "min_p", "n_p_lt_1e5", "n_p_lt_1e6", "n_p_lt_5e8",
        "lead_chr", "lead_pos", "lead_id"
    ]].sort_values("min_p").head(30)
)

total ok phenotypes: 1532
phenotypes with min_p < 1e-5 : 1532
phenotypes with min_p < 1e-6 : 1252
phenotypes with min_p < 5e-8 : 125
phenotypes with >=1 hit p<1e-5: 1532
phenotypes with >=1 hit p<1e-6: 1252
phenotypes with >=1 hit p<5e-8: 125


,phenotype,min_p,n_p_lt_1e5,n_p_lt_1e6,n_p_lt_5e8,lead_chr,lead_pos,lead_id
0,PE_18_2_22_2,5.459910e-10,336,197,82,8,114969281,chr8_114969281
1,PC_20_2_22_4,9.325720e-10,103,15,5,3,3078681,chr3_3078681
2,TAG52_2_FA20_0,1.236100e-09,83,15,6,21,31858140,chr21_31858140
3,PE_18_1_18_4,1.690420e-09,172,70,4,5,142589362,chr5_142589362
4,P_3_Phosphoglyceric_acid,1.996020e-09,198,84,8,2,153646482,chr2_153646482
5,TAG58_7_FA22_5,2.190780e-09,106,49,1,8,72746337,chr8_72746337
6,PE_16_1_20_4,2.367650e-09,129,36,10,10,96153621,chr10_96153621
7,PE_16_0_16_0,2.738120e-09,98,13,11,20,63119632,chr20_63119632
8,TAG54_4_FA18_2,2.773820e-09,22,2,1,11,65246264,chr11_65246264
9,L_Methionine,3.087140e-09,153,35,10,14,50978200,chr14_50978200


## Step 5. 生成“值得画图”的候选表

### 一级优先（strong）

满足任一：

- `min_p < 5e-8`
- 或 `n_p_lt_5e8 >= 2`

### 二级优先（suggestive）

在不属于 `strong` 的前提下，满足任一：

- `min_p < 1e-7`
- 或 `n_p_lt_5e8 >= 1`

这套更合理，因为：

- `5e-8` 还是标准 genome-wide significant
- `n_p_lt_5e8 >= 2` 代表不是孤立单点
- `min_p < 1e-7` 留住差一点但很接近 genome-wide 的
- `n_p_lt_5e8 >= 1` 可以保留至少有一个 genome-wide hit 的 phenotype

这比拿 `1e-5 / 1e-6` 的计数阈值稳很多。

In [9]:
cand_df = ok_df.copy()

if "file_size_mb_exact" not in cand_df.columns and "file_size_mb" in cand_df.columns:
    cand_df["file_size_mb_exact"] = cand_df["file_size_mb"]

cand_df["plot_priority"] = "low"

# 一级优先（strong）
cand_df.loc[
    (cand_df["min_p"] < 5e-8) |
    (cand_df["n_p_lt_5e8"].fillna(0) >= 2),
    "plot_priority"
] = "strong"

# 二级优先（suggestive）
cand_df.loc[
    cand_df["plot_priority"].eq("low") & (
        (cand_df["min_p"] < 1e-7) |
        (cand_df["n_p_lt_5e8"].fillna(0) >= 1)
    ),
    "plot_priority"
] = "suggestive"

priority_order = {"strong": 2, "suggestive": 1, "low": 0}
cand_df["plot_priority_rank"] = cand_df["plot_priority"].map(priority_order)

cand_df = cand_df.sort_values(
    ["plot_priority_rank", "min_p", "n_p_lt_5e8", "n_p_lt_1e6"],
    ascending=[False, True, False, False]
).drop(columns="plot_priority_rank").reset_index(drop=True)

cand_df.to_csv(PLOT_CANDIDATE_TSV, sep="\t", index=False)

top_df = ok_df.sort_values("min_p").head(100).copy()
top_df.to_csv(TOP_PHENO_TSV, sep="\t", index=False)

print("saved:", PLOT_CANDIDATE_TSV)
print("saved:", TOP_PHENO_TSV)

print("\nplot_priority counts:")
print(cand_df["plot_priority"].value_counts(dropna=False))

print("\nsize by priority (MB):")
print(
    cand_df.groupby("plot_priority", dropna=False)["file_size_mb_exact"]
    .sum()
    .sort_values(ascending=False)
)

display(cand_df[[
    "phenotype", "plot_priority", "min_p",
    "n_p_lt_1e5", "n_p_lt_1e6", "n_p_lt_5e8",
    "lead_chr", "lead_pos", "lead_id"
]].head(50))

saved: /mnt/zzbnew/peixunban/zhanghebin/CIMA_multiomics_regulation/data/results/gwas_all/summary/add_plot_candidates.tsv
saved: /mnt/zzbnew/peixunban/zhanghebin/CIMA_multiomics_regulation/data/results/gwas_all/summary/add_top_phenotypes_by_minp.tsv

plot_priority counts:
plot_priority
low           1297
strong         125
suggestive     110
Name: count, dtype: int64

size by priority (MB):
plot_priority
low           893541.8708
strong         86040.0634
suggestive     75840.3935
Name: file_size_mb_exact, dtype: float64


,phenotype,plot_priority,min_p,n_p_lt_1e5,n_p_lt_1e6,n_p_lt_5e8,lead_chr,lead_pos,lead_id
0,PE_18_2_22_2,strong,5.459910e-10,336,197,82,8,114969281,chr8_114969281
1,PC_20_2_22_4,strong,9.325720e-10,103,15,5,3,3078681,chr3_3078681
2,TAG52_2_FA20_0,strong,1.236100e-09,83,15,6,21,31858140,chr21_31858140
3,PE_18_1_18_4,strong,1.690420e-09,172,70,4,5,142589362,chr5_142589362
4,P_3_Phosphoglyceric_acid,strong,1.996020e-09,198,84,8,2,153646482,chr2_153646482
5,TAG58_7_FA22_5,strong,2.190780e-09,106,49,1,8,72746337,chr8_72746337
6,PE_16_1_20_4,strong,2.367650e-09,129,36,10,10,96153621,chr10_96153621
7,PE_16_0_16_0,strong,2.738120e-09,98,13,11,20,63119632,chr20_63119632
8,TAG54_4_FA18_2,strong,2.773820e-09,22,2,1,11,65246264,chr11_65246264
9,L_Methionine,strong,3.087140e-09,153,35,10,14,50978200,chr14_50978200


## Step 6. 看 failed 文件

如果有 `.add` 读不动，先定位。

In [10]:
failed_df = summary_df[summary_df["status"] != "ok"].copy()
print("failed count:", len(failed_df))
display(failed_df.head(20))

failed count: 0


,phenotype,file,is_gz,exists,file_size_mb,status,error,n_rows,n_non_na_p,min_p,...,lead_id,lead_beta,lead_se,lead_errcode,n_p_lt_1e5,n_p_lt_1e6,n_p_lt_5e8,p01,p001,p0001


## Step 7. 空间层面的决策建议

先看这几个数字再决定保留策略：

- 全部 ADD 占多少
- 强/建议候选的 ADD 占多少

In [11]:
def safe_size_mb(path_str):
    p = Path(path_str)
    return p.stat().st_size / 1024 / 1024 if p.exists() else 0

ok_df["file_size_mb_exact"] = ok_df["file"].map(safe_size_mb)

size_all_mb = ok_df["file_size_mb_exact"].sum()
size_strong_mb = ok_df.loc[ok_df["plot_priority"].eq("strong"), "file_size_mb_exact"].sum() if "plot_priority" in ok_df.columns else 0
size_sugg_mb = cand_df.loc[cand_df["plot_priority"].isin(["strong", "suggestive"]), "file_size_mb_exact"].sum()

print(f"all ADD size: {size_all_mb:.2f} MB")
print(f"strong only size: {size_strong_mb:.2f} MB")
print(f"strong+suggestive size: {size_sugg_mb:.2f} MB")

all ADD size: 1055422.33 MB
strong only size: 0.00 MB
strong+suggestive size: 161880.46 MB


## Step 8. 可选：只保留候选 phenotype 的 ADD，其他移动到 `.legacy`

**默认先不执行。**
先看上面 summary，确认后再解除注释运行。

说明：

- 这里不直接删，而是移动到 `.legacy/gwas_add_non_candidate/`
- 更安全
- 只处理 `.add` / `.add.gz`

In [ ]:
# ===== 可选执行块：先不要直接跑 =====

from pathlib import Path
import shutil

LEGACY_DIR = PROJECT_ROOT / ".legacy" / "gwas_add_non_candidate"
LEGACY_DIR.mkdir(parents=True, exist_ok=True)

# 可选保留范围：
# keep_mode = "strong"
# keep_mode = "strong_and_suggestive"
keep_mode = None   # 先保持 None，不执行

if keep_mode is not None:
    keep_set = set()
    if keep_mode == "strong":
        keep_set = set(cand_df.loc[cand_df["plot_priority"] == "strong", "phenotype"])
    elif keep_mode == "strong_and_suggestive":
        keep_set = set(cand_df.loc[cand_df["plot_priority"].isin(["strong", "suggestive"]), "phenotype"])
    else:
        raise ValueError("unknown keep_mode")

    moved = []
    for _, row in ok_df.iterrows():
        phenotype = row["phenotype"]
        src = Path(row["file"])
        if phenotype in keep_set:
            continue

        rel = src.relative_to(PROJECT_ROOT)
        dst = LEGACY_DIR / rel
        dst.parent.mkdir(parents=True, exist_ok=True)

        if src.exists():
            shutil.move(str(src), str(dst))
            moved.append({
                "phenotype": phenotype,
                "src": str(src),
                "dst": str(dst)
            })

    moved_df = pd.DataFrame(moved)
    display(moved_df.head())
    print("moved count:", len(moved_df))
else:
    print("未执行移动。先看 summary 再决定。")

## Step 9. 可选：对保留的 ADD 压缩成 `.gz`

这个步骤建议在你决定好保留哪些 phenotype 之后再做。

In [ ]:
# ===== 可选执行块：先不要直接跑 =====

import gzip
import shutil
from pathlib import Path

def gzip_file_remove_src(src: Path):
    dst = Path(str(src) + ".gz")
    with open(src, "rb") as fin, gzip.open(dst, "wb") as fout:
        shutil.copyfileobj(fin, fout)
    if dst.exists() and dst.stat().st_size > 0:
        src.unlink()
    return dst

RUN_GZIP = False

if RUN_GZIP:
    target_files = []
    for f in PER_PHENO_DIR.glob("*/*.glm.linear.add"):
        if f.is_file():
            target_files.append(f)

    print("to gzip:", len(target_files))
    done = []
    for i, f in enumerate(target_files, 1):
        dst = gzip_file_remove_src(f)
        done.append({"src": str(f), "dst": str(dst)})
        if i % 50 == 0 or i == len(target_files):
            print(f"[{i}/{len(target_files)}] gzipped")

    gzip_df = pd.DataFrame(done)
    display(gzip_df.head())
else:
    print("未执行 gzip。建议在筛选完保留文件后再压缩。")

## 建议你实际怎么做

### 最稳方案
1. 跑完整个 notebook
2. 看 `add_plot_candidates.tsv`
3. 先只保留：
   - `plot_priority == strong`
   - 或 `strong + suggestive`
4. 再对保留文件 gzip

### 为什么不建议只保留 genome-wide
因为你样本量不大，很多真实线索可能还在 `1e-5 ~ 1e-6` 这个区间。  
这些表型非常适合优先画 Manhattan / QQ 图，而不是直接删掉。